# YOLO26 Medium — GC10-DET Steel Defect Detection

**목적**: Colab T4 GPU에서 YOLO26 Medium 모델 학습 → ONNX export → Google Drive 저장

| 항목 | 값 |
|------|----|
| 모델 | YOLO26m |
| 데이터셋 | GC10-DET (10 classes, 1835 train / 459 val) |
| imgsz | 1024 |
| 비교 대상 | Nano (mAP50=0.701), Small (mAP50=0.720) |

> **Runtime → Change runtime type → T4 GPU** 확인 후 실행

## 1. 환경 설정

In [ ]:
# GPU 확인
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

In [ ]:
# Ultralytics 설치
!pip install -q ultralytics>=8.4.0

import ultralytics
print(f"Ultralytics: {ultralytics.__version__}")
ultralytics.checks()

In [ ]:
# Google Drive 마운트 (결과 저장용)
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUT = '/content/drive/MyDrive/yolo26-results'
os.makedirs(DRIVE_OUT, exist_ok=True)
print(f"Drive output: {DRIVE_OUT}")

## 2. GC10-DET 데이터셋 준비

GC10-DET(Galvanized Steel Sheet Defect Detection) — 10종 강판 결함, 2,294장

In [ ]:
%%time
# GC10-DET 다운로드 (GitHub release)
!pip install -q gdown
import gdown

# GC10-DET Supervisely format
url = 'https://github.com/lvxiaoming2002/GC10-DET/archive/refs/heads/main.zip'
!wget -q {url} -O /content/gc10-raw.zip
!unzip -q /content/gc10-raw.zip -d /content/gc10-raw/
print("GC10-DET raw data downloaded")

import os
raw_path = '/content/gc10-raw'
# 구조 확인
for root, dirs, files in os.walk(raw_path):
    depth = root.replace(raw_path, '').count(os.sep)
    if depth < 3:
        indent = ' ' * 2 * depth
        print(f"{indent}{os.path.basename(root)}/")
        if depth == 2:
            print(f"{indent}  ({len(files)} files)")

In [ ]:
import json
import os
import shutil
import random
from pathlib import Path

# === GC10-DET Supervisely JSON -> YOLO 변환 ===
# 원본 경로 (다운로드 구조에 따라 조정 필요)
RAW_BASE = Path('/content/gc10-raw')
# GitHub zip 구조: GC10-DET-main/GC10-DET/
candidates = list(RAW_BASE.rglob('ann'))
if candidates:
    RAW_ANN = candidates[0]
    RAW_IMG = RAW_ANN.parent / 'images'
    print(f"Found ann: {RAW_ANN}")
    print(f"Found img: {RAW_IMG}")
else:
    raise FileNotFoundError("ann 폴더를 찾을 수 없습니다. 압축 구조를 확인하세요.")

OUT = Path('/content/datasets/gc10-det')

CLASS_NAMES = [
    'crease', 'crescent_gap', 'inclusion', 'oil_spot', 'punching_hole',
    'rolled_pit', 'silk_spot', 'waist_folding', 'water_spot', 'welding_line'
]
CLASS_MAP = {name: idx for idx, name in enumerate(CLASS_NAMES)}
TRAIN_RATIO = 0.8
random.seed(42)

OUT.mkdir(parents=True, exist_ok=True)
for split in ['train', 'val']:
    (OUT / 'images' / split).mkdir(parents=True, exist_ok=True)
    (OUT / 'labels' / split).mkdir(parents=True, exist_ok=True)

samples = []
ann_files = sorted(RAW_ANN.glob('*.json'))
print(f"Found {len(ann_files)} annotation files")

for ann_file in ann_files:
    with open(ann_file, 'r') as f:
        data = json.load(f)

    img_h = data['size']['height']
    img_w = data['size']['width']
    img_name = ann_file.stem

    # 이미지 파일 찾기
    img_path = None
    for ext in ['.jpg', '.jpeg', '.png', '.bmp']:
        candidate = RAW_IMG / (img_name + ext)
        if candidate.exists():
            img_path = candidate
            break
    if img_path is None:
        continue

    labels = []
    for obj in data.get('objects', []):
        cls_name = obj['classTitle']
        if cls_name not in CLASS_MAP:
            continue
        cls_id = CLASS_MAP[cls_name]
        pts = obj['points']['exterior']
        x1, y1 = pts[0]
        x2, y2 = pts[1]
        cx = ((x1 + x2) / 2) / img_w
        cy = ((y1 + y2) / 2) / img_h
        bw = abs(x2 - x1) / img_w
        bh = abs(y2 - y1) / img_h
        labels.append(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

    samples.append((img_path, labels))

random.shuffle(samples)
split_idx = int(len(samples) * TRAIN_RATIO)
splits = {'train': samples[:split_idx], 'val': samples[split_idx:]}

total_objs = 0
for split, items in splits.items():
    for img_path, labels in items:
        shutil.copy2(img_path, OUT / 'images' / split / img_path.name)
        lbl_path = OUT / 'labels' / split / (img_path.stem + '.txt')
        with open(lbl_path, 'w') as f:
            f.write('\n'.join(labels))
        total_objs += len(labels)
    print(f"{split}: {len(items)} images")

print(f"Total objects: {total_objs}")

# data.yaml 생성
yaml_content = f"""path: {OUT}
train: images/train
val: images/val

nc: 10
names: {CLASS_NAMES}
"""
with open(OUT / 'data.yaml', 'w') as f:
    f.write(yaml_content)

print(f"\ndata.yaml saved to {OUT / 'data.yaml'}")
print("Dataset ready!")

## 3. YOLO26 Medium 학습

로컬 Nano/Small v3와 동일한 하이퍼파라미터 사용 (공정 비교)

In [ ]:
from ultralytics import YOLO

# YOLO26 Medium 모델 로드
model = YOLO('yolo26m.pt')

# 모델 정보 확인
print(f"Model: {model.model_name}")
print(f"Parameters: {sum(p.numel() for p in model.model.parameters()):,}")

In [ ]:
%%time
# 학습 시작 — Nano/Small v3와 동일 하이퍼파라미터
results = model.train(
    data='/content/datasets/gc10-det/data.yaml',
    epochs=200,
    imgsz=1024,
    batch=4,              # T4 15GB에서 안정적인 batch
    device=0,
    project='/content/results',
    name='yolo26m_gc10det_v3',
    exist_ok=True,
    patience=30,
    cos_lr=True,
    copy_paste=0.2,
    scale=0.7,
    degrees=5,
    mixup=0.1,
    close_mosaic=15,
    pretrained=True,
    optimizer='auto',
    workers=2,            # Colab은 Linux라 workers 사용 가능
    verbose=True,
)

print("\n" + "=" * 60)
print("Training Complete!")
print("=" * 60)

## 4. 결과 확인

In [ ]:
import pandas as pd
from IPython.display import display

# 학습 결과 CSV 로드
csv_path = '/content/results/yolo26m_gc10det_v3/results.csv'
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()

# Best epoch
best_idx = df['metrics/mAP50(B)'].idxmax()
best = df.iloc[best_idx]

print("=" * 50)
print(f"Best Epoch: {int(best['epoch'])}")
print(f"mAP50:      {best['metrics/mAP50(B)']:.5f}")
print(f"mAP50-95:   {best['metrics/mAP50-95(B)']:.5f}")
print(f"Precision:  {best['metrics/precision(B)']:.5f}")
print(f"Recall:     {best['metrics/recall(B)']:.5f}")
print("=" * 50)

# Nano/Small과 비교
print("\n--- Nano vs Small vs Medium (GC10-DET v3) ---")
print(f"{'Model':<10} {'mAP50':<10} {'Params':<12} {'Note'}")
print(f"{'Nano':<10} {'0.701':<10} {'2.5M':<12} {'Local RTX 3060'}")
print(f"{'Small':<10} {'0.720':<10} {'9.96M':<12} {'Local RTX 3060'}")
print(f"{'Medium':<10} {best['metrics/mAP50(B)']:<10.3f} {'~25M':<12} {'Colab T4'}")

In [ ]:
# 학습 곡선 시각화
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# mAP50
axes[0].plot(df['epoch'], df['metrics/mAP50(B)'], 'b-', linewidth=1.5)
axes[0].axhline(y=0.701, color='g', linestyle='--', alpha=0.7, label='Nano v3 (0.701)')
axes[0].axhline(y=0.720, color='orange', linestyle='--', alpha=0.7, label='Small v3 (0.720)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('mAP50')
axes[0].set_title('mAP50 (Medium vs Nano/Small baseline)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(df['epoch'], df['train/box_loss'], label='box_loss')
axes[1].plot(df['epoch'], df['train/cls_loss'], label='cls_loss')
axes[1].plot(df['epoch'], df['train/dfl_loss'], label='dfl_loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Training Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# P/R
axes[2].plot(df['epoch'], df['metrics/precision(B)'], label='Precision')
axes[2].plot(df['epoch'], df['metrics/recall(B)'], label='Recall')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Score')
axes[2].set_title('Precision / Recall')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/results/yolo26m_gc10det_v3/training_curves.png', dpi=150)
plt.show()
print("Training curves saved")

## 5. ONNX Export

In [ ]:
from ultralytics import YOLO

# best.pt 로드 → ONNX export
best_model = YOLO('/content/results/yolo26m_gc10det_v3/weights/best.pt')
best_model.export(format='onnx', imgsz=1024)

import os
onnx_path = '/content/results/yolo26m_gc10det_v3/weights/best.onnx'
onnx_size = os.path.getsize(onnx_path) / (1024 * 1024)
print(f"\nONNX exported: {onnx_size:.1f} MB")

## 6. Google Drive에 저장

In [ ]:
import shutil

# 핵심 파일만 Drive에 복사
files_to_save = [
    ('weights/best.pt', 'yolo26m_gc10det_v3_best.pt'),
    ('weights/best.onnx', 'yolo26m_gc10det_v3_best.onnx'),
    ('results.csv', 'yolo26m_gc10det_v3_results.csv'),
    ('args.yaml', 'yolo26m_gc10det_v3_args.yaml'),
    ('training_curves.png', 'yolo26m_gc10det_v3_curves.png'),
]

src_dir = '/content/results/yolo26m_gc10det_v3'
for src, dst in files_to_save:
    src_path = os.path.join(src_dir, src)
    dst_path = os.path.join(DRIVE_OUT, dst)
    if os.path.exists(src_path):
        shutil.copy2(src_path, dst_path)
        size = os.path.getsize(dst_path) / (1024 * 1024)
        print(f"  Saved: {dst} ({size:.1f} MB)")
    else:
        print(f"  SKIP: {src} (not found)")

print(f"\nAll files saved to Google Drive: {DRIVE_OUT}")
print("\n=== ONNX 파일을 로컬 PC로 다운로드해서 RTX 3060에서 추론 벤치마크 ===\n")
print("다음 단계:")
print("1. Google Drive에서 best.onnx 다운로드")
print("2. 로컬 PC: python benchmark_inference.py")
print("3. Nano vs Small vs Medium 속도/정확도 비교")

## 7. (선택) Large 모델 학습

T4에서 가능하지만 batch=1~2로 제한됨. OOM 발생 시 imgsz=640으로 변경.

⚠️ **Pro/Pro+ A100이면 batch=4로 편하게 가능**

In [ ]:
# === Large 모델 (선택 실행) ===
# T4에서 실행 시 batch=1~2, OOM 가능성 있음
# A100이면 batch=4 가능

TRAIN_LARGE = False  # True로 변경하면 실행

if TRAIN_LARGE:
    model_l = YOLO('yolo26l.pt')
    print(f"Large params: {sum(p.numel() for p in model_l.model.parameters()):,}")
    
    results_l = model_l.train(
        data='/content/datasets/gc10-det/data.yaml',
        epochs=200,
        imgsz=1024,         # OOM 시 640으로 변경
        batch=2,            # T4: 2가 한계. A100: 4~8 가능
        device=0,
        project='/content/results',
        name='yolo26l_gc10det_v3',
        exist_ok=True,
        patience=30,
        cos_lr=True,
        copy_paste=0.2,
        scale=0.7,
        degrees=5,
        mixup=0.1,
        close_mosaic=15,
        pretrained=True,
        workers=2,
    )
    
    # ONNX export
    best_l = YOLO('/content/results/yolo26l_gc10det_v3/weights/best.pt')
    best_l.export(format='onnx', imgsz=1024)
    
    # Drive 저장
    for src, dst in [
        ('weights/best.pt', 'yolo26l_gc10det_v3_best.pt'),
        ('weights/best.onnx', 'yolo26l_gc10det_v3_best.onnx'),
        ('results.csv', 'yolo26l_gc10det_v3_results.csv'),
    ]:
        sp = f'/content/results/yolo26l_gc10det_v3/{src}'
        if os.path.exists(sp):
            shutil.copy2(sp, os.path.join(DRIVE_OUT, dst))
            print(f"  Saved: {dst}")
else:
    print("Large training skipped. Set TRAIN_LARGE = True to run.")

---

### 다음 단계 (로컬 PC에서)

```python
# Google Drive에서 best.onnx 다운로드 후:
# C:\dev\active\yolo26-industrial-vision\results\yolo26m_gc10det_v3\weights\best.onnx
#
# RTX 3060에서 추론 벤치마크:
# python scripts/benchmark_inference.py
#
# 비교 결과표:
# | Model  | mAP50 | Params | ONNX Size | GPU Infer | CPU Infer |
# |--------|-------|--------|-----------|-----------|----------|
# | Nano   | 0.701 | 2.5M   | 9.6 MB    | ~3ms      | ~50ms    |
# | Small  | 0.720 | 9.96M  | 36.7 MB   | ~5ms      | ~120ms   |
# | Medium | ???   | ~25M   | ~80 MB    | ~8ms      | ~300ms   |
```